# 4. Governance and reports

Every result should travel with a record of how it was made. This tutorial writes a run manifest, a provenance
document, a frozen specification with a deviation record, a TRIPOD+AI checklist and an HTML report.

In [1]:
from pathlib import Path

from symphony_dt.data import simulate_cohort
from symphony_dt.governance import ProvenanceDocument, RunManifest

out = Path("tutorial_governance")
out.mkdir(exist_ok=True)
cohort = simulate_cohort(50, seed=9)
manifest = RunManifest.create("simulate", model="pd_v2", data_sha256=cohort.stan_data.sha256(),
                              dataset_fingerprint=cohort.dataset.fingerprint(), seeds={"simulate": 9})
manifest.write(out / "manifest.json")
{k: v for k, v in manifest.to_dict().items() if k in ("kind", "package_version", "dataset_fingerprint")}

{'dataset_fingerprint': '57204672287e5d3fe3a7d01393a892ca43b0ccf235425c612475194df7965a17',
 'kind': 'simulate',
 'package_version': '2.0.0'}

In [2]:
doc = ProvenanceDocument.for_run("simulate", {}, {"stan_data.json": cohort.stan_data.sha256()})
doc.write(out / "provenance.json")
list(doc.to_dict())

['prefix',
 'entity',
 'activity',
 'agent',
 'used',
 'wasGeneratedBy',
 'wasAssociatedWith']

## Pre-registration

In [3]:
from symphony_dt.errors import UnfrozenSpecificationError
from symphony_dt.evaluate import DeviationRecord, ScoringSpec, freeze_spec, require_frozen

spec = ScoringSpec.default()
frozen = freeze_spec(spec, out / "frozen_spec.json")
changed = spec.model_copy(update={"min_bin": 20})
try:
    require_frozen(changed, frozen)
except UnfrozenSpecificationError as err:
    print("refused:", err)
require_frozen(changed, frozen, DeviationRecord("smaller cohort", "A. Analyst", {"min_bin": 20}))["deviation"]

refused: the specification changed after it was frozen (ed849af62b66 to 8d26abff7640); write a DeviationRecord to proceed


{'reason': 'smaller cohort',
 'author': 'A. Analyst',
 'changes': {'min_bin': 20},
 'recorded_at': '2026-11-06T08:34:45.290477+00:00'}

## TRIPOD+AI checklist

In [4]:
from symphony_dt.governance import tripod_ai_report

print(tripod_ai_report({"title": "Title page", "code_sharing": "Methods, code availability"})[:600])

# TRIPOD+AI checklist

Source: Collins GS, Moons KGM, Dhiman P, et al. TRIPOD+AI statement: updated guidance for reporting clinical prediction models that use regression or machine learning methods. BMJ. 2024;385:e078378.

| heading | prompt | where reported |
|---|---|---|
| Title | Identify the study as developing or evaluating a prediction model, the population and the outcome. | Title page |
| Abstract | Give a structured summary following the TRIPOD+AI abstract guidance. | TO DO |
| Background | Explain the clinical context and why a model is needed. | TO DO |
| Objectives | State whether


## HTML report

In [5]:
from symphony_dt.report import write_html_report

write_html_report(out / "report.html", [manifest.to_dict()], title="Synthetic run",
                  notes=["Synthetic data only."])

PosixPath('tutorial_governance/report.html')